In [ ]:
#!/usr/bin/env python3
"""
AI Java Tutor — ML Predictive Modeling Pipeline
================================================
Target: quiz_percentage (binned into performance categories)
Approach: PCA + Logistic Regression (following user's Titanic template),
          plus Random Forest and SVM for comparison.
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, f1_score)
import warnings
warnings.filterwarnings('ignore')

# ─── Color palette (matching poster) ───
NAVY = '#0B1D3A'
DEEP = '#132B50'
ACCENT = '#F59E0B'
ACCENT_L = '#FCD34D'
TEAL = '#14B8A6'
TEAL_L = '#5EEAD4'
CORAL = '#F87171'
WHITE = '#FAFBFC'
GRAY4 = '#94A3B8'
GRAY6 = '#475569'

plt.rcParams.update({
    'figure.facecolor': NAVY,
    'axes.facecolor': DEEP,
    'axes.edgecolor': GRAY6,
    'axes.labelcolor': GRAY4,
    'xtick.color': GRAY4,
    'ytick.color': GRAY4,
    'text.color': WHITE,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.grid': True,
    'grid.color': '#1E3A5F',
    'grid.alpha': 0.5,
})

OUT = '/home/claude'

# ═══════════════════════════════════════════════════════════════
# 1. LOAD & PREPARE DATA
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("1. LOADING DATA")
print("=" * 60)
data_dir = "./data/"
df = pd.read_csv(data_dir + 'sessions_with_engagement_features_updated.csv')
print(f"   Raw shape: {df.shape}")

# --- Encode categoricals ---
df['session_type_enc'] = LabelEncoder().fit_transform(df['session_type'])  # arraylist=0, recursion=1
df['has_both_enc'] = df['has_both'].astype(int)

# --- Fill nulls ---
# avg_difficulty_incorrect has 26 nulls (students who got everything right) → fill with 0
df['avg_difficulty_incorrect'] = df['avg_difficulty_incorrect'].fillna(0)
# avg_difficulty_correct has 2 nulls (students who got nothing right) → fill with 0
df['avg_difficulty_correct'] = df['avg_difficulty_correct'].fillna(0)
# Response time cols: 3 nulls → fill with median
resp_cols = ['avg_response_time', 'median_response_time', 'std_response_time',
             'min_response_time', 'max_response_time', 'rapid_response_count', 'rapid_response_pct']
for c in resp_cols:
    df[c] = df[c].fillna(df[c].median())

# --- Feature selection ---
feature_cols = [
    'condition', 'session_type_enc', 'duration_seconds',
    'total_messages', 'user_messages', 'assistant_messages',
    'avg_response_time', 'median_response_time', 'std_response_time',
    'min_response_time', 'max_response_time',
    'rapid_response_count', 'rapid_response_pct',
    'has_both_enc', 'avg_difficulty_correct', 'avg_difficulty_incorrect'
]

# --- Target: Bin quiz_percentage into categories ---
# Binary: High (>=70) vs Low (<70)
df['quiz_label'] = (df['quiz_percentage'] >= 70).astype(int)
print(f"   Target distribution (High≥70 vs Low<70):")
print(f"   {df['quiz_label'].value_counts().to_dict()}")

# Also create 3-class for exploration
df['quiz_class3'] = pd.cut(df['quiz_percentage'], bins=[-1, 40, 70, 100],
                           labels=['Low', 'Mid', 'High'])

X = df[feature_cols].copy()
y = df['quiz_label'].copy()

print(f"   Features: {len(feature_cols)}")
print(f"   Samples:  {len(X)}")

# ═══════════════════════════════════════════════════════════════
# 2. SCALING + PCA
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("2. STANDARD SCALING + PCA")
print("=" * 60)

scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)

# Full PCA to analyze variance
pca_full = PCA(random_state=44)
pca_full.fit(X_scaled)

cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = np.argmax(cum_var >= 0.95) + 1
n_components_90 = np.argmax(cum_var >= 0.90) + 1
print(f"   Components for 90% variance: {n_components_90}")
print(f"   Components for 95% variance: {n_components_95}")

# --- PCA Scree Plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Individual variance
bars = ax1.bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
               pca_full.explained_variance_ratio_, color=TEAL, alpha=0.8, edgecolor=TEAL_L, linewidth=0.5)
ax1.set_xlabel('Principal Component', fontsize=12)
ax1.set_ylabel('Explained Variance Ratio', fontsize=12)
ax1.set_title('PCA Scree Plot', fontsize=16, fontweight='bold', color=ACCENT)

# Cumulative variance
ax2.plot(range(1, len(cum_var) + 1), cum_var, color=ACCENT, linewidth=2.5, marker='o', markersize=5)
ax2.axhline(y=0.90, color=CORAL, linestyle='--', linewidth=1, alpha=0.7, label='90% threshold')
ax2.axhline(y=0.95, color=TEAL_L, linestyle='--', linewidth=1, alpha=0.7, label='95% threshold')
ax2.axvline(x=n_components_90, color=CORAL, linestyle=':', linewidth=1, alpha=0.5)
ax2.axvline(x=n_components_95, color=TEAL_L, linestyle=':', linewidth=1, alpha=0.5)
ax2.set_xlabel('Number of Components', fontsize=12)
ax2.set_ylabel('Cumulative Variance', fontsize=12)
ax2.set_title('Cumulative Explained Variance', fontsize=16, fontweight='bold', color=ACCENT)
ax2.legend(facecolor=DEEP, edgecolor=GRAY6, labelcolor=GRAY4)

plt.tight_layout()
plt.savefig(f'{OUT}/pca_scree.png', dpi=150, bbox_inches='tight', facecolor=NAVY)
plt.close()
print("   ✓ Saved pca_scree.png")

# --- PCA 2D Scatter ---
pca_2d = PCA(n_components=2, random_state=44)
X_pca2 = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
colors_map = {0: CORAL, 1: TEAL}
for label, color in colors_map.items():
    mask = y == label
    name = 'High (≥70%)' if label == 1 else 'Low (<70%)'
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=color, label=name,
               alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)', fontsize=13)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)', fontsize=13)
ax.set_title('PCA — Student Sessions by Quiz Performance', fontsize=16, fontweight='bold', color=ACCENT)
ax.legend(facecolor=DEEP, edgecolor=GRAY6, labelcolor=WHITE, fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT}/pca_scatter.png', dpi=150, bbox_inches='tight', facecolor=NAVY)
plt.close()
print("   ✓ Saved pca_scatter.png")

# --- PCA Loadings ---
pca_fit = PCA(n_components=n_components_95, random_state=44)
X_pca = pca_fit.fit_transform(X_scaled)

loadings = pd.DataFrame(
    pca_fit.components_.T,
    columns=[f'PC{i+1}' for i in range(n_components_95)],
    index=feature_cols
)

fig, ax = plt.subplots(figsize=(12, 7))
# Show loadings for first 3 PCs
load_plot = loadings.iloc[:, :3]
x_pos = np.arange(len(feature_cols))
width = 0.25
ax.barh(x_pos - width, load_plot.iloc[:, 0], width, color=ACCENT, alpha=0.85, label='PC1')
ax.barh(x_pos, load_plot.iloc[:, 1], width, color=TEAL, alpha=0.85, label='PC2')
ax.barh(x_pos + width, load_plot.iloc[:, 2], width, color=CORAL, alpha=0.85, label='PC3')
ax.set_yticks(x_pos)
ax.set_yticklabels([c.replace('_', ' ').title() for c in feature_cols], fontsize=9)
ax.set_xlabel('Loading', fontsize=12)
ax.set_title('PCA Loadings (Top 3 Components)', fontsize=16, fontweight='bold', color=ACCENT)
ax.legend(facecolor=DEEP, edgecolor=GRAY6, labelcolor=WHITE)
ax.axvline(x=0, color=GRAY6, linewidth=0.8)
plt.tight_layout()
plt.savefig(f'{OUT}/pca_loadings.png', dpi=150, bbox_inches='tight', facecolor=NAVY)
plt.close()
print("   ✓ Saved pca_loadings.png")

# ═══════════════════════════════════════════════════════════════
# 3. MODEL TRAINING (following user's Titanic pattern)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("3. MODEL TRAINING")
print("=" * 60)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=44, stratify=y
)
print(f"   Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

# Also prepare raw (non-PCA) scaled version for comparison
X_train_raw, X_test_raw, _, _ = train_test_split(
    X_scaled, y, test_size=0.2, random_state=44, stratify=y
)

models = {
    'Logistic Regression (PCA)': LogisticRegression(max_iter=1000, random_state=44),
    'Logistic Regression (Raw)': LogisticRegression(max_iter=1000, random_state=44),
    'Random Forest (PCA)': RandomForestClassifier(n_estimators=100, random_state=44, max_depth=5),
    'Gradient Boosting (PCA)': GradientBoostingClassifier(n_estimators=100, random_state=44, max_depth=3),
    'SVM (PCA)': SVC(kernel='rbf', random_state=44, probability=True),
}

results = {}
for name, model in models.items():
    if 'Raw' in name:
        model.fit(X_train_raw, y_train)
        train_pred = model.predict(X_train_raw)
        test_pred = model.predict(X_test_raw)
        test_proba = model.predict_proba(X_test_raw)[:, 1] if hasattr(model, 'predict_proba') else None
        # Cross-val on raw
        cv_scores = cross_val_score(model, X_scaled, y, cv=StratifiedKFold(5, shuffle=True, random_state=44), scoring='accuracy')
    else:
        model.fit(X_train, y_train)
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        test_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
        cv_scores = cross_val_score(model, X_pca, y, cv=StratifiedKFold(5, shuffle=True, random_state=44), scoring='accuracy')

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred, average='weighted')
    auc = roc_auc_score(y_test, test_proba) if test_proba is not None else None

    results[name] = {
        'train_acc': train_acc, 'test_acc': test_acc,
        'test_f1': test_f1, 'auc': auc,
        'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
        'test_pred': test_pred, 'test_proba': test_proba
    }

    print(f"\n   {name}:")
    print(f"     Train Accuracy: {train_acc:.4f}")
    print(f"     Test Accuracy:  {test_acc:.4f}")
    print(f"     Test F1:        {test_f1:.4f}")
    if auc: print(f"     AUC-ROC:        {auc:.4f}")
    print(f"     5-Fold CV:      {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# ═══════════════════════════════════════════════════════════════
# 4. VISUALIZATIONS
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("4. GENERATING VISUALIZATIONS")
print("=" * 60)

# --- Model Comparison Bar Chart ---
fig, ax = plt.subplots(figsize=(12, 6))
model_names = list(results.keys())
short_names = ['LogReg\n(PCA)', 'LogReg\n(Raw)', 'Random\nForest', 'Gradient\nBoosting', 'SVM\n(PCA)']
x = np.arange(len(model_names))
width = 0.3

train_accs = [results[n]['train_acc'] for n in model_names]
test_accs = [results[n]['test_acc'] for n in model_names]
cv_means = [results[n]['cv_mean'] for n in model_names]

bars1 = ax.bar(x - width, train_accs, width, color=TEAL, alpha=0.85, label='Train Acc')
bars2 = ax.bar(x, test_accs, width, color=ACCENT, alpha=0.85, label='Test Acc')
bars3 = ax.bar(x + width, cv_means, width, color=CORAL, alpha=0.85, label='CV Mean')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.008, f'{h:.2f}',
                ha='center', va='bottom', fontsize=9, color=WHITE)

ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=10)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('Model Comparison: Train vs Test vs Cross-Validation', fontsize=16, fontweight='bold', color=ACCENT)
ax.set_ylim(0, 1.12)
ax.legend(facecolor=DEEP, edgecolor=GRAY6, labelcolor=WHITE, fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUT}/model_comparison.png', dpi=150, bbox_inches='tight', facecolor=NAVY)
plt.close()
print("   ✓ Saved model_comparison.png")

# --- Confusion Matrices ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_models = ['Logistic Regression (PCA)', 'Random Forest (PCA)', 'Gradient Boosting (PCA)']
for i, name in enumerate(plot_models):
    cm = confusion_matrix(y_test, results[name]['test_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr', ax=axes[i],
                xticklabels=['Low', 'High'], yticklabels=['Low', 'High'],
                cbar_kws={'label': 'Count'}, linewidths=0.5, linecolor=GRAY6)
    axes[i].set_title(name.replace(' (PCA)', ''), fontsize=13, fontweight='bold', color=ACCENT)
    axes[i].set_xlabel('Predicted', fontsize=11)
    axes[i].set_ylabel('Actual', fontsize=11)

plt.suptitle('Confusion Matrices — Test Set', fontsize=16, fontweight='bold', color=WHITE, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor=NAVY)
plt.close()
print("   ✓ Saved confusion_matrices.png")

# --- Feature Importance (Random Forest on raw features) ---
rf_raw = RandomForestClassifier(n_estimators=200, random_state=44, max_depth=5)
rf_raw.fit(X_train_raw, y_train)
importances = pd.Series(rf_raw.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = [ACCENT if v >= importances.quantile(0.75) else TEAL for v in importances]
ax.barh(range(len(importances)), importances.values, color=colors, edgecolor='none', height=0.7)
ax.set_yticks(range(len(importances)))
ax.set_yticklabels([c.replace('_', ' ').title() for c in importances.index], fontsize=10)
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Random Forest Feature Importance', fontsize=16, fontweight='bold', color=ACCENT)
plt.tight_layout()
plt.savefig(f'{OUT}/feature_importance.png', dpi=150, bbox_inches='tight', facecolor=NAVY)
plt.close()
print("   ✓ Saved feature_importance.png")

# --- Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(14, 10))
corr_cols = feature_cols + ['quiz_percentage']
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=ax, linewidths=0.5, linecolor=NAVY, annot_kws={'size': 8},
            xticklabels=[c.replace('_', '\n').title() for c in corr_cols],
            yticklabels=[c.replace('_', ' ').title() for c in corr_cols])
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', color=ACCENT)
plt.tight_layout()
plt.savefig(f'{OUT}/correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor=NAVY)
plt.close()
print("   ✓ Saved correlation_heatmap.png")

# --- PCA by Condition ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cond_names = {1: 'Persona Scaffolded', 2: 'Non-Persona Scaffolded', 3: 'Direct Chat'}
cond_colors = {1: TEAL, 2: ACCENT, 3: CORAL}
for i, cond in enumerate([1, 2, 3]):
    mask_cond = df['condition'] == cond
    for label, color in {0: '#F87171', 1: '#5EEAD4'}.items():
        mask = mask_cond & (y == label)
        name = 'High' if label == 1 else 'Low'
        axes[i].scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=color, label=name,
                       alpha=0.7, s=50, edgecolors='white', linewidth=0.5)
    axes[i].set_title(cond_names[cond], fontsize=13, fontweight='bold', color=cond_colors[cond])
    axes[i].set_xlabel('PC1', fontsize=10)
    axes[i].set_ylabel('PC2', fontsize=10)
    axes[i].legend(facecolor=DEEP, edgecolor=GRAY6, labelcolor=WHITE, fontsize=9)

plt.suptitle('PCA Clusters by Instructional Condition', fontsize=16, fontweight='bold', color=WHITE, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/pca_by_condition.png', dpi=150, bbox_inches='tight', facecolor=NAVY)
plt.close()
print("   ✓ Saved pca_by_condition.png")

# ═══════════════════════════════════════════════════════════════
# 5. PREDICTIONS OUTPUT
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("5. SAVING PREDICTIONS")
print("=" * 60)

# Best model predictions on full dataset
best_model_name = max(results, key=lambda k: results[k]['cv_mean'])
print(f"   Best model (by CV): {best_model_name}")

# Refit best model on all data for predictions
if 'Raw' in best_model_name:
    best_model = models[best_model_name].__class__(**models[best_model_name].get_params())
    best_model.fit(X_scaled, y)
    all_pred = best_model.predict(X_scaled)
    all_proba = best_model.predict_proba(X_scaled)[:, 1]
else:
    best_model = models[best_model_name].__class__(**models[best_model_name].get_params())
    best_model.fit(X_pca, y)
    all_pred = best_model.predict(X_pca)
    all_proba = best_model.predict_proba(X_pca)[:, 1]

output_df = pd.DataFrame({
    'user_id': df['user_id'],
    'session_type': df['session_type'],
    'condition': df['condition'].astype(int),
    'quiz_percentage': df['quiz_percentage'],
    'predicted_label': all_pred,
    'predicted_probability_high': all_proba.round(4)
})
output_df.to_csv(f'{OUT}/ml_predictions.csv', index=None)
print("   ✓ Saved ml_predictions.csv")

# ═══════════════════════════════════════════════════════════════
# 6. SUMMARY REPORT
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("6. SUMMARY")
print("=" * 60)

summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Train Acc': [f"{results[n]['train_acc']:.3f}" for n in results],
    'Test Acc': [f"{results[n]['test_acc']:.3f}" for n in results],
    'Test F1': [f"{results[n]['test_f1']:.3f}" for n in results],
    'AUC-ROC': [f"{results[n]['auc']:.3f}" if results[n]['auc'] else 'N/A' for n in results],
    'CV Mean': [f"{results[n]['cv_mean']:.3f}" for n in results],
    'CV Std': [f"{results[n]['cv_std']:.3f}" for n in results],
})
print(summary.to_string(index=False))

print(f"\n   PCA Components used: {n_components_95} (95% variance)")
print(f"   Best model: {best_model_name} (CV: {results[best_model_name]['cv_mean']:.3f})")
print(f"\n   Top 5 features (RF importance):")
for feat, imp in importances.tail(5).items():
    print(f"     • {feat.replace('_', ' ').title()}: {imp:.4f}")

print("\n" + "=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)